In [11]:
import os.path
from glob import glob
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from PIL import Image
Image.MAX_IMAGE_PIXELS = 1e11

In [13]:
pwd

'/Users/sagana/Library/CloudStorage/OneDrive-UniversityofPittsburgh/code0/SpaTraFact/data/Andersson2021'

In [4]:
!mkdir ../../data/Andersson2021
%cd ../../data/Andersson2021
!git clone https://github.com/almaan/her2st.git


mkdir: ../../data/Andersson2021: File exists
/Users/sagana/Library/CloudStorage/OneDrive-UniversityofPittsburgh/code0/SpaTraFact/data/Andersson2021
fatal: destination path 'her2st' already exists and is not an empty directory.


In [5]:
!mkdir props/
!unzip her2st/res/ST-deconv/props/major.zip -d props/
!unzip her2st/res/ST-deconv/props/minor.zip -d props/
!unzip her2st/res/ST-deconv/props/subset.zip -d props/

mkdir: props/: File exists
Archive:  her2st/res/ST-deconv/props/major.zip
replace props/major/D2-proportion.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C
Archive:  her2st/res/ST-deconv/props/minor.zip
replace props/minor/D2-proportion.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C
Archive:  her2st/res/ST-deconv/props/subset.zip
replace props/subset/D2-proportion.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


In [16]:
sample_list=[x[-9:-7] for x in glob("her2st/data/ST-cnts/*.tsv.gz")]
adatas={a: dict() for a in np.unique([x[0] for x in sample_list])}

for sample in sample_list:
        adata=sc.read_csv("her2st/data/St-cnts/"+sample+".tsv.gz", delimiter=None)
        adata.layers["counts"]=adata.X
        adata.obs_names=sample+'_'+adata.obs_names

        coord=pd.read_csv("her2st/data/ST-spotfiles/"+sample+"_selection.tsv", delimiter="\t")
        coord.index=[sample+'_'+str(coord.iloc[i,0])+"x"+str(coord.iloc[i,1]) for i in coord.index]
        adata.obsm['spatial']=coord.loc[adata.obs.index,["pixel_x","pixel_y"]].values
        adata.obsm['spatial_int']=coord.loc[adata.obs.index,["x","y"]].values

        for level in ["major", "minor", "subset"]:
                df=pd.read_csv("props/"+level+"/"+sample+"-proportion.tsv", index_col=0, delimiter='\t')
                df.index=adata.obs_names
                adata.obsm["celltype_"+level]=df

        adata.obs["Total Reads"]=np.sum(adata.X, axis=1)
        impath=glob("her2st/data/ST-imgs/"+sample[0]+"/"+sample+"/*.jpg")[0]

        adata.uns={}
        adata.uns['spatial']={}
        adata.uns['spatial'][sample]={}
        adata.uns['spatial'][sample]['images']={}
        adata.uns['spatial'][sample]['scalefactors']={}
        adata.uns['spatial'][sample]['images']['hires']= np.asarray(Image.open(impath))
        adata.uns['spatial'][sample]['scalefactors']={'spot_diameter_fullres': 130,'tissue_hires_scalef': 1,'fiducial_diameter_fullres': 280}


        pat_file="her2st/data/ST-pat/lbl/"+sample+"_labeled_coordinates.tsv"
        if os.path.exists(pat_file):
                pat_df=pd.read_csv(pat_file, sep="\t").dropna()
                pat_df.index=[sample+"_"+str(round(x))+"x"+str(round(y)) for (x,y) in zip(pat_df["x"], pat_df["y"])]
                pat_df=pat_df.loc[adata.obs_names]

                adata.obs["pathology"]=pat_df["label"]
        else:
                adata.obs["pathology"]=None

        adata.obs["sample"]=sample[0]
        adata.obs["replicate"]=sample
        adata.obs["source"]="Andersson2021"
        adata.obs["ER"]=False #all samples are ER-
        adata.obs["HER2"]=True #all samples are HER2+
        adata.obs["PR"]= "B" in sample #all are PR- with the exception of patient B
        adata.obs["subtype"]="HER2"

        adata.layers["raw_counts"]=adata.X
        adata.write_h5ad(sample+".h5ad")
        adatas[sample[0]][sample[1]]=adata



... storing 'sample' as categorical
... storing 'replicate' as categorical
... storing 'source' as categorical
... storing 'subtype' as categorical
... storing 'pathology' as categorical
... storing 'sample' as categorical
... storing 'replicate' as categorical
... storing 'source' as categorical
... storing 'subtype' as categorical
... storing 'sample' as categorical
... storing 'replicate' as categorical
... storing 'source' as categorical
... storing 'subtype' as categorical
... storing 'sample' as categorical
... storing 'replicate' as categorical
... storing 'source' as categorical
... storing 'subtype' as categorical
... storing 'sample' as categorical
... storing 'replicate' as categorical
... storing 'source' as categorical
... storing 'subtype' as categorical
... storing 'pathology' as categorical
... storing 'sample' as categorical
... storing 'replicate' as categorical
... storing 'source' as categorical
... storing 'subtype' as categorical
... storing 'sample' as categorica

In [15]:
sample_list

[]